# Evaluation pipeline demo


This is a short notebook showing how the evaluation pipeline works WITHOUT having to download the entire dataset and configure the container. Small subsets of each dataset is included in duckdb files in this folder.

Ensure you have Python 3 and `uv`, then just run `uv sync` from the repository root and ensure the notebook is connected to that virtual environment (likely `.venv` by default)

## Datasets
1. [Risk-Based Authentication (RBA)](https://zenodo.org/records/6782156) (Wiefling et al, 2022)
    - synthetic dataset of 31m authentication records from ~3m users with user IDs from an SSO service
    - includes user agent, IP, etc., as well as fields for (a) whether the login originates from a known attack IP address or (b) the login was flagged as an account takeover (ground truth)
    - Licensed by the authors (Wiefling et al.) under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

2. [FP-Stalker](https://github.com/Spirals-Team/FPStalker) (Vastel et al., 2018) 
    - 15k record sample from the AmIUnique dataset from 2015-2017. 
    - Each record contains a long term tracking cookie (tying it to other records from the same origin) alongside a full user agent string and other fingerprint attributes we do not use in our ER method (e.g., canvas).
    - Licensed by the authors (Vastel et al.) under the [GNU-AGPL 3.0 License](https://github.com/Spirals-Team/FPStalker/blob/master/LICENSE)



The datasets provided in this demo are small subsets of the above datasets just to show how this works at a high level:
1. RBA subset (3.2k rows): Selected only only "User IDs" that have at least one row flagged as "Is Account Takeover"
2. FP Stalker Subset (8.6k rows): Selected only tracking cookie IDs that were live in June 2016. Also removed some columns that aren't useful to us.


In [ ]:
import duckdb
import pandas as pd

with duckdb.connect('./trunc_rba.duckdb') as rba_conn:
    rba_df_orig = rba_conn.execute("SELECT * FROM imported_data").df()

print("Length of RBA truncated dataset:", rba_df_orig.shape[0])
display(rba_df_orig.head())

In [ ]:
with duckdb.connect('./trunc_fp_stalker.duckdb') as fp_conn:
    fp_df_orig = fp_conn.execute("SELECT * FROM imported_data").df() 

print("Length of FP Stalker truncated dataset:", fp_df_orig.shape[0])
display(fp_df_orig.head())

# Parse User Agent Strings 

Parse the user agent strings (e.g., `Mozilla/5.0 (Windows NT 6.1; WOW64; rv:41.0) Gecko/20100101 Firefox/41.0`) using the same `field_normalization` modules as the webapp, then move them into the column names expected by the `device_grouping2` module.

In [ ]:
import sys
from pathlib import Path
import tqdm

_root = Path.cwd().parent.parent
_python_core = str(_root / "python_core")
if _python_core not in sys.path:
    sys.path.append(_python_core)
if str(_root) not in sys.path:
    sys.path.append(str(_root))

from python_core.field_normalization.user_agent import UserAgentParser
from python_core.field_normalization.device import normalize_device_fields

In [ ]:
ua_parser = UserAgentParser()

def normalize_data(df: pd.DataFrame, 
                   user_agent_col_name: str, 
                   timestamp_col_name: str,
                   other_cols_to_keep: list = []) -> pd.DataFrame:
    dicts = []
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        d = {"user_agent_original": row.get(user_agent_col_name, "")}
        d = ua_parser.parse(d) # returns dict
        d = normalize_device_fields(d)
        d = {k: v for k, v in d.items() if k.startswith("norm__")}
        d['timestamp'] = row.get(timestamp_col_name, "")
        dicts.append(d)
    
    new_columns = pd.DataFrame(dicts)
    normalized_cols = [c for c in new_columns.columns if c.startswith("norm__")]
    
    if isinstance(other_cols_to_keep, list):
        other_cols_to_keep = [c for c in other_cols_to_keep if c in df.columns]
    else:
        other_cols_to_keep = []
    
    new_df = df[other_cols_to_keep + [user_agent_col_name]]
    new_df = pd.concat([
        new_columns[["timestamp"] + normalized_cols], 
        new_df], axis=1)
    new_df.rename(columns={c: f"attr__{c}" for c in normalized_cols}, inplace=True)
    return new_df

In [ ]:
s = "Mozilla/5.0  (iPhone; CPU iPhone OS 11_2_6 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko Version/4.0 Chrome/85.0.4183.127 Mobile Safari/537.36Snapchat11.1.7.81 (LYA-L29; Android 10#10.1.0.288C432#29; gzip"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/69.0.3497.17.19.92 Safari/537.36"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15"
s = "Mozilla/5.0  (iPad; CPU OS 7_1 like Mac OS X) AppleWebKit/533.1 (KHTML, like Gecko Version/4.0 Mobile Safari/533.1 variation/277457"
ua_parser.parse({"user_agent_original": s})

In [ ]:
norm_rba_df = normalize_data(rba_df_orig,
                             user_agent_col_name='User Agent String', 
                             timestamp_col_name='Login Timestamp',
                             other_cols_to_keep=['index', 'User ID', 'Login Successful', 'Is Account Takeover', 'Is Attack IP', 'Browser Name and Version', 'OS Name and Version'])                           

norm_rba_df.rename(columns={'index': 'id'},  inplace=True)  # "id" has to be the unique id of the row
norm_rba_df.convert_dtypes()
norm_rba_df

In [ ]:
norm_fp_df = normalize_data(fp_df_orig,
                            user_agent_col_name='userAgentHttp', 
                            timestamp_col_name='creationDate',
                            other_cols_to_keep=['id', 'counter', 'osDetailed', 'browserDetailed'])

norm_fp_df.rename(columns={'id': 'tracking_id'},  inplace=True)  # "id" has to be the unique id of the row
norm_fp_df.rename(columns={'counter': 'id'}, inplace=True)

norm_fp_df.convert_dtypes()
norm_fp_df

# RBA Heuristic Evaluation

We evaluate our device grouping heuristic using the dataset from:
> Stephan Wiefling, Paul René Jørgensen, Sigurd Thunem, and Luigi Lo Iacono. 2022. **Pump Up Password Security! Evaluating and Enhancing Risk-Based Authentication on a Real-World Large-Scale Online Service**. *ACM Transactions on Privacy and Security (TOPS)*. [DOI: 10.1145/3549079](https://doi.org/10.1145/3549079) | [Zenodo Dataset](https://doi.org/10.5281/zenodo.6782155)

The dataset contains synthetic values for privacy. Each record includes:
- User account ID (`User ID`)
- User agent and metadata fields (`User Agent String`)
- Account takeover flag (`Is Account Takeover` - ground-truth compromise)
- Known attack IP flag (`Is Attack IP` - ground-truth indicator of compromise)

For all user accounts with at least one flagged account takeover, we group their logins into device instances and evaluate:
1. Do we accidentally group benign logins with takeover logins?
2. Are these errors caused by spoofed user agents matching benign ones?

In [ ]:
from python_core.device_grouping2 import client_os_upgrades, profiles
from python_core.device_grouping2.instances import DeviceInstanceGraph
import datetime
import json

In [ ]:
start_time  = datetime.datetime.now().isoformat()

rba_results = {
    "run": {
        "description": "Truncated RBA dataset for demo",
        "start_time": start_time,
    },
    "summary": {},
    "computed": []
}

# -----------

for user_id in map(int, norm_rba_df['User ID'].unique()):
    user_df = norm_rba_df[norm_rba_df['User ID'] == user_id]

    row_ids_with_account_takeover = set(user_df[user_df['Is Account Takeover'] == True]['id'].tolist())
    row_ids_with_attack_ip = set(user_df[user_df['Is Attack IP'] == True]['id'].tolist())

    upgrade_edges = client_os_upgrades.get_edges(user_df) # build up edges between records
    graph = DeviceInstanceGraph(user_df, pd.DataFrame(upgrade_edges)) # construct subgraphs from all the edges

    for i, inst in enumerate(graph.get_instances()):
        dct = {}
        dct["user_id"] = user_id
        dct["instance_number"] = i

        dct["instance_metadata"] = inst.export_as_dict()
        
        records_with_account_takeover = list(row_ids_with_account_takeover.intersection(set(inst.df['id'].tolist())))
        records_with_attack_ip = list(row_ids_with_attack_ip.intersection(set(inst.df['id'].tolist())))
        
        non_takeover_records = list(set(inst.df['id'].tolist()).difference(set(records_with_account_takeover)))
        non_attack_ip_records = list(set(inst.df['id'].tolist()).difference(set(records_with_attack_ip)))
        
        dct['record_ids'] = {
            "all": inst.df['id'].tolist(),
            "account_takeover": records_with_account_takeover,
            "non_takeover": non_takeover_records,
            "attack_ip": records_with_attack_ip,
            "non_attack_ip": non_attack_ip_records,
        }

        dct['summary'] = {
            "num_records": len(dct['record_ids']),
            "num_records_with_account_takeover": len(records_with_account_takeover),
            "num_records_non_takeover": len(non_takeover_records),
            "num_records_with_attack_ip": len(records_with_attack_ip),
            "num_records_non_attack_ip": len(non_attack_ip_records),
        }

        # UA matching analysis for takeover vs non-takeover
        takeover_uas = set(inst.df[inst.df['id'].isin(records_with_account_takeover)]['User Agent String'].dropna())
        non_takeover_uas = set(inst.df[inst.df['id'].isin(non_takeover_records)]['User Agent String'].dropna())
        has_identical_ua_takeover = len(takeover_uas.intersection(non_takeover_uas)) > 0

        # UA matching analysis for attack vs non-attack
        attack_uas = set(inst.df[inst.df['id'].isin(records_with_attack_ip)]['User Agent String'].dropna())
        non_attack_uas = set(inst.df[inst.df['id'].isin(non_attack_ip_records)]['User Agent String'].dropna())
        has_identical_ua_attack = len(attack_uas.intersection(non_attack_uas)) > 0

        dct['analysis'] = {
            "has_identical_ua_takeover": has_identical_ua_takeover,
            "has_identical_ua_attack": has_identical_ua_attack,
            "takeover_uas": list(takeover_uas),
            "non_takeover_uas": list(non_takeover_uas),
            "attack_uas": list(attack_uas),
            "non_attack_uas": list(non_attack_uas),
        }

        rba_results["computed"].append(dct)


In [ ]:
# compute summary 
num_singleton = 0

# Takeover stats
num_takeover_inst = 0
num_only_non_takeover = 0
num_only_takeover = 0
num_mixed_takeover = 0
takeover_mixed_instances = []

# Attack IP stats
num_attack_ip_inst = 0
num_only_non_attack_ip = 0
num_only_attack_ip = 0
num_mixed_attack_ip = 0
attack_mixed_instances = []

sizes = []

for inst in rba_results["computed"]:
    r_ids = inst['record_ids']
    sz = len(r_ids['all'])
    sizes.append(sz)
    if sz == 1: num_singleton += 1
        
    has_takeover = len(r_ids['account_takeover']) > 0
    has_non_takeover = len(r_ids['non_takeover']) > 0
    
    has_attack = len(r_ids['attack_ip']) > 0
    has_non_attack = len(r_ids['non_attack_ip']) > 0
    
    # Takeover evaluation
    if has_takeover:
        num_takeover_inst += 1
        if has_non_takeover:
            num_mixed_takeover += 1
            takeover_mixed_instances.append(inst)
        else:
            num_only_takeover += 1
    elif has_non_takeover:
        num_only_non_takeover += 1

    # Attack IP evaluation
    if has_attack:
        num_attack_ip_inst += 1
        if has_non_attack:
            num_mixed_attack_ip += 1
            attack_mixed_instances.append(inst)
        else:
            num_only_attack_ip += 1
    elif has_non_attack:
        num_only_non_attack_ip += 1

s_sizes = pd.Series(sizes) if sizes else pd.Series([0])

rba_results["summary"] = {
    "num_records": len(norm_rba_df),
    "num_users": len(norm_rba_df['User ID'].unique()),
    "num_instances": len(rba_results["computed"]),
    "instance_sizes": {
        "num_singleton_instances": num_singleton,
        "median_instance_size": int(s_sizes.median()),
        "mean_instance_size": float(s_sizes.mean()),
        "max_instance_size": int(s_sizes.max()),
    },
    "account_takeover_stats": {
        "num_records_with_account_takeover": int(norm_rba_df['Is Account Takeover'].sum()),
        "num_instances_with_account_takeover": num_takeover_inst,
        "num_instances_only_non_takeover": num_only_non_takeover,
        "num_instances_only_takeover": num_only_takeover,
        "num_instances_mixed_takeover": num_mixed_takeover,
        "num_instances_mixed_takeover_with_identical_ua": sum(1 for inst in takeover_mixed_instances if inst['analysis']['has_identical_ua_takeover']),
    },
    "attack_ip_stats": {
        "num_records_with_attack_ip": int(norm_rba_df['Is Attack IP'].sum()),
        "num_instances_with_attack_ip": num_attack_ip_inst,
        "num_instances_only_non_attack_ip": num_only_non_attack_ip,
        "num_instances_only_attack_ip": num_only_attack_ip,
        "num_instances_mixed_attack_ip": num_mixed_attack_ip,
        "num_instances_mixed_attack_ip_with_identical_ua": sum(1 for inst in attack_mixed_instances if inst['analysis']['has_identical_ua_attack']),
    }
}

In [ ]:
# make dir /rundata
_start_time = start_time.replace(":", "-").replace(".", "-")
dirpath =  Path("runs") / f"rba_{_start_time}"
dirpath.mkdir(parents=True, exist_ok=True)
with open(dirpath / "rba_results.json", "w") as f:
    json.dump(rba_results, f, indent=4)
norm_rba_df.to_csv(dirpath / "rba_results.csv", index=False)

with open(dirpath / "takeout_and_benign_instances.json", "w") as f:
    json.dump(takeover_mixed_instances, f, indent=4)
